In [ ]:
import albumentations as A 
import cv2 
import glob 
import math
import numpy as np 
import os 
import pandas as pd

from tqdm import tqdm

In [ ]:
ROOT = "/mnt/stor/datasets/CBIS-DDSM/"

In [ ]:
df = pd.read_csv(os.path.join(ROOT, "train_kfold_cv.csv"))
df.head()

In [ ]:
imgfiles = glob.glob(os.path.join(ROOT, "cropped_png", "*.png"))
imgfiles[:2]

In [ ]:
def tile_image_into_patches(img, mask, patch_ratio):
    patch_size = round(patch_ratio * img.shape[0])
    new_width = math.ceil(img.shape[1] / patch_size) * patch_size
    img = np.pad(img, ((0, 0), (0, new_width - img.shape[1])), "constant")
    mask = np.pad(mask, ((0, 0), (0, new_width - mask.shape[1])), "constant")
    patches = {"image": [], "mask": []}
    for xi in range(0, img.shape[0] - patch_size + 1, patch_size):
        for yi in range(0, img.shape[1] - patch_size + 1, patch_size):
            cropped_img = img[xi : xi + patch_size, yi : yi + patch_size]
            cropped_mask = mask[xi : xi + patch_size, yi : yi + patch_size]
            patches["image"].append(cropped_img)
            patches["mask"].append(cropped_mask)
    return patches

In [ ]:
PATCH_RATIO = 0.125

img_save_dir = os.path.join(ROOT, "patches_1-8")
mask_save_dir = os.path.join(ROOT, "patches_masks_1-8")

os.makedirs(img_save_dir, exist_ok=True)
os.makedirs(mask_save_dir, exist_ok=True)

for f in tqdm(imgfiles):
    img = cv2.imread(f, 0)
    mask = cv2.imread(f.replace("png/", "masks/"), 0)
    patches = tile_image_into_patches(img, mask, PATCH_RATIO)

    for i, p in enumerate(patches["image"]):
        cv2.imwrite(
            os.path.join(img_save_dir, f"{os.path.basename(f).replace('.png', '')}_{i:03d}.png"), p
        )
    
    for i, p in enumerate(patches["mask"]):
        cv2.imwrite(
            os.path.join(mask_save_dir, f"{os.path.basename(f).replace('.png', '')}_{i:03d}.png"), p
        )

In [ ]:
2744 / 686

In [ ]:
print(len(patches["image"]))

In [ ]:
import matplotlib.pyplot as plt 


for p in patches["image"]:
    plt.imshow(p, cmap="gray")
    plt.show()